# 2. Feature Engineering
- This notebook focuses on transforming raw engine sensor data into model-ready features, including RUL target generation and temporal feature engineering.


In [36]:
# import raw dataset 
import pandas as pd

columns = ['engine_id', 'cycle'] + \
          [f'op_setting_{i}' for i in range(1, 4)] + \
          [f'sensor_{i}' for i in range(1, 22)]
df = pd.read_csv(
    r'..\data\raw\train_FD001.txt',
    sep=r'\s+', 
    header=None,
    names=columns,
    engine='python'
)

In [37]:
df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [38]:
df = df.dropna(axis=1)

In [39]:
# sorting the data
df = df.sort_values(["engine_id", "cycle"]).reset_index(drop=True)

### Create RUL

In [40]:
# max cycle per engine
max_cycles = df.groupby("engine_id")["cycle"].max()

In [41]:
df["RUL"] = df["engine_id"].map(max_cycles) - df["cycle"]

In [42]:
df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


In [43]:
# dropping non informative columns
df = df.drop(columns=[f'op_setting_{i}' for i in range(1, 4)])

In [44]:
sensor_cols = [col for col in df.columns if "sensor_" in col]

### Rolling window feature
- This turn raw sensor data into predictive signals.

In [46]:
WINDOW = 20

In [47]:
df_fe = df.copy()

for sensor in sensor_cols:
    df_fe[f"{sensor}_roll_mean"] = (
        df_fe.groupby("engine_id")[sensor]
        .rolling(WINDOW, min_periods=1).mean().reset_index(level=0, drop=True))

    df_fe[f"{sensor}_roll_std"] = (
        df_fe.groupby("engine_id")[sensor]
        .rolling(WINDOW, min_periods=1).std()
        .reset_index(level=0, drop=True)
    )

In [48]:
df_fe.head()

,engine_id,cycle,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,...,sensor_17_roll_mean,sensor_17_roll_std,sensor_18_roll_mean,sensor_18_roll_std,sensor_19_roll_mean,sensor_19_roll_std,sensor_20_roll_mean,sensor_20_roll_std,sensor_21_roll_mean,sensor_21_roll_std
0,1,1,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,...,392.000000,NaN,2388.0,NaN,100.0,NaN,39.060000,NaN,23.419000,NaN
1,1,2,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,...,392.000000,0.000000,2388.0,0.0,100.0,0.0,39.030000,0.042426,23.421300,0.003253
2,1,3,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,...,391.333333,1.154701,2388.0,0.0,100.0,0.0,39.003333,0.055076,23.395600,0.044573
3,1,4,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,...,391.500000,1.000000,2388.0,0.0,100.0,0.0,38.972500,0.076322,23.390175,0.037977
4,1,5,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,...,391.800000,1.095445,2388.0,0.0,100.0,0.0,38.958000,0.073621,23.393020,0.033498


In [49]:
df_fe = df_fe.fillna(0)

In [50]:
df_fe.filter(like="sensor_2").head(10)

,sensor_2,sensor_20,sensor_21,sensor_2_roll_mean,sensor_2_roll_std,sensor_20_roll_mean,sensor_20_roll_std,sensor_21_roll_mean,sensor_21_roll_std
0,641.82,39.06,23.4190,641.820000,0.000000,39.060000,0.000000,23.419000,0.000000
1,642.15,39.00,23.4236,641.985000,0.233345,39.030000,0.042426,23.421300,0.003253
2,642.35,38.95,23.3442,642.106667,0.267644,39.003333,0.055076,23.395600,0.044573
3,642.35,38.88,23.3739,642.167500,0.250117,38.972500,0.076322,23.390175,0.037977
4,642.37,38.90,23.4044,642.208000,0.234776,38.958000,0.073621,23.393020,0.033498
5,642.10,38.98,23.3669,642.190000,0.214569,38.961667,0.066458,23.388667,0.031803
6,642.48,39.10,23.3774,642.231429,0.224457,38.981429,0.080089,23.387057,0.029343
7,642.56,38.97,23.3106,642.272500,0.238073,38.980000,0.074258,23.377500,0.038324
8,642.12,39.05,23.4066,642.255556,0.228425,38.987778,0.073276,23.380733,0.037138
9,641.71,38.95,23.4694,642.201000,0.275941,38.984000,0.070111,23.389600,0.044857


In [51]:
df_fe.head()

,engine_id,cycle,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,...,sensor_17_roll_mean,sensor_17_roll_std,sensor_18_roll_mean,sensor_18_roll_std,sensor_19_roll_mean,sensor_19_roll_std,sensor_20_roll_mean,sensor_20_roll_std,sensor_21_roll_mean,sensor_21_roll_std
0,1,1,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,...,392.000000,0.000000,2388.0,0.0,100.0,0.0,39.060000,0.000000,23.419000,0.000000
1,1,2,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,...,392.000000,0.000000,2388.0,0.0,100.0,0.0,39.030000,0.042426,23.421300,0.003253
2,1,3,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,...,391.333333,1.154701,2388.0,0.0,100.0,0.0,39.003333,0.055076,23.395600,0.044573
3,1,4,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,...,391.500000,1.000000,2388.0,0.0,100.0,0.0,38.972500,0.076322,23.390175,0.037977
4,1,5,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,...,391.800000,1.095445,2388.0,0.0,100.0,0.0,38.958000,0.073621,23.393020,0.033498


In [63]:
df_fe.columns


Index(['engine_id', 'cycle', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4',
       'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10',
       'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
       'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20',
       'sensor_21', 'RUL', 'sensor_1_roll_mean', 'sensor_1_roll_std',
       'sensor_2_roll_mean', 'sensor_2_roll_std', 'sensor_3_roll_mean',
       'sensor_3_roll_std', 'sensor_4_roll_mean', 'sensor_4_roll_std',
       'sensor_5_roll_mean', 'sensor_5_roll_std', 'sensor_6_roll_mean',
       'sensor_6_roll_std', 'sensor_7_roll_mean', 'sensor_7_roll_std',
       'sensor_8_roll_mean', 'sensor_8_roll_std', 'sensor_9_roll_mean',
       'sensor_9_roll_std', 'sensor_10_roll_mean', 'sensor_10_roll_std',
       'sensor_11_roll_mean', 'sensor_11_roll_std', 'sensor_12_roll_mean',
       'sensor_12_roll_std', 'sensor_13_roll_mean', 'sensor_13_roll_std',
       'sensor_14_roll_mean', 'sensor_14_roll_std

### Preparaing processed dataset

In [52]:
# drop non feature columns
drop_cols = ["engine_id", "cycle"]
df_model = df_fe.drop(columns=drop_cols)

In [53]:
from sklearn.model_selection import train_test_split

In [54]:
engine_ids = df["engine_id"].unique()

In [55]:
train_engines, test_engines = train_test_split(engine_ids, test_size=0.2, random_state=42)

In [56]:
train_df = df_fe[df_fe["engine_id"].isin(train_engines)]
test_df  = df_fe[df_fe["engine_id"].isin(test_engines)]

In [57]:
X_train = train_df.drop(columns=["engine_id", "cycle", "RUL"])
y_train = train_df["RUL"]

X_test = test_df.drop(columns=["engine_id", "cycle", "RUL"])
y_test = test_df["RUL"]

In [58]:
X_train.shape

(16561, 63)

In [59]:
y_train.shape

(16561,)

In [60]:
X_test.shape

(4070, 63)

In [61]:
y_test.shape

(4070,)

In [62]:
# saving final csv file 
df_fe.to_csv("../data/processed/processed_data.csv",index=False)